# Step 0 - Data preparation (80 speakers)
Runtime setup first: **Runtime > Change runtime type > T4 GPU (or A100/L4 if offered), Runtime shape > High-RAM if offered (not required)** (Colab Pro). Then run every cell top to bottom with **Run all**.

Holdout (unseen) speakers are always p263-p266. Shards are saved to Drive, so this notebook can be resumed.

In [ ]:
!pip install -q datasets faster-whisper librosa soundfile num2words

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

ROOT = "/content/drive/MyDrive/voice_cloning_v2"   # folder that contains vc_lib/
import sys, torch
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE -> Runtime > Change runtime type > GPU")

## Load CUDA libraries for Whisper (needed in every new session)

In [ ]:
!pip install -q nvidia-cublas-cu12 "nvidia-cudnn-cu12==9.*"
import ctypes, glob, os
libs = []
for pkg in ["cublas", "cudnn"]:
    for d in glob.glob(f"/usr/local/lib/python3*/dist-packages/nvidia/{pkg}/lib"):
        libs += [os.path.join(d, f) for f in os.listdir(d) if ".so" in f]
pending = sorted(libs)
while pending:
    failed = []
    for lib in pending:
        try:
            ctypes.CDLL(lib, mode=ctypes.RTLD_GLOBAL)
        except OSError:
            failed.append(lib)
    if len(failed) == len(pending):
        print("Could not load:", [os.path.basename(f) for f in failed]); break
    pending = failed
print("CUDA libraries loaded for Whisper.")

## Build the speaker shards (~4-5 h for 80 speakers, resumable)

In [ ]:
from vc_lib import prep

prep.prepare_shards(ROOT, num_speakers=80, num_holdout=4, whisper_size="small",
                    whisper_device="cuda", whisper_compute="float16")

## Build the training cache (measures the dB range, splits by sentence)

In [ ]:
from vc_lib import prep
meta = prep.build_cache(ROOT, crops_per_utt=1)
meta["db_range"], meta["counts"], meta["chunk_chars"]

## Sanity check: listen to random segments (the text above each clip should match)

In [ ]:
import random, librosa.display, matplotlib.pyplot as plt
from IPython.display import Audio, display
from vc_lib.common import paths, load_cache, segment_audio, from_int16, SR
C = load_cache(paths(ROOT))
for i in random.sample(range(len(C['texts'])), 4):
    print(C['spks'][i], C['split'][i], '|', C['texts'][i])
    display(Audio(segment_audio(paths(ROOT), C, i), rate=SR))
plt.figure(figsize=(6, 3)); librosa.display.specshow(C['mels'][i].float().numpy(), sr=SR, hop_length=256, x_axis='time', y_axis='mel')
plt.title(C['texts'][i]); plt.colorbar(); plt.show()

## Optional: Whisper check of the cuts (about 3 min on CPU; prints progress every 10 clips)

In [ ]:
!pip install -q faster-whisper
import random, numpy as np, librosa
from faster_whisper import WhisperModel
from vc_lib.common import paths, load_cache, segment_audio, from_int16, SR
from vc_lib.infer import word_error_rate
if "C" not in globals():
    C = load_cache(paths(ROOT))
wh = WhisperModel("base", device="cpu", compute_type="int8")
random.seed(1)
res = []
for n, i in enumerate(random.sample(range(len(C["texts"])), 60), 1):
    y = segment_audio(paths(ROOT), C, i)
    segs, _ = wh.transcribe(librosa.resample(y, orig_sr=SR, target_sr=16000), language="en", beam_size=1)
    hyp = " ".join(s.text for s in segs).strip()
    res.append((word_error_rate(C["texts"][i], hyp), i, hyp))
    if n % 10 == 0:
        w = np.array([r[0] for r in res])
        print(f"{n} clips | mean WER {w.mean():.2f} | perfect {np.mean(w == 0):.0%} | badly cut {np.mean(w >= 0.5):.0%}")